# 00 · Setup and data audit
Recompute the binary corpus inventory from decoded PCM. Do not write under the raw data root. This notebook reads the multiclass folder only for its one-time identity check.

In [ ]:
from pathlib import Path
import sys, json, hashlib, numpy as np, pandas as pd
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT/'src'))
from dronedet import data
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
with (PROJECT / 'config.yaml').open(encoding='utf-8') as f: CFG = __import__('yaml').safe_load(f)
profile = 'colab' if 'google.colab' in sys.modules else 'local'
P = {k: Path(v) for k, v in CFG['paths'][profile].items()}
ROOT = data.locate_dataset_root(P['raw_root'])
OUT = PROJECT / 'data'
def safe(path):
    path = Path(path).resolve(); raw = P['raw_root'].resolve()
    if raw == path or raw in path.parents: raise PermissionError(f'refuse raw write: {path}')
    path.parent.mkdir(parents=True, exist_ok=True); return path
FROZEN=(PROJECT/'data/processed/splits/FROZEN.txt').exists()
# Post-freeze verification decodes training clips only.
ids = pd.read_csv(PROJECT/'data/processed/splits/train.csv',keep_default_na=False).clip_id.tolist() if FROZEN else data.list_binary_wavs(ROOT)
inv = data.build_inventory(ROOT, ids)
inv['label'] = (inv.folder == 'yes_drone').astype('int8')
parsed = pd.DataFrame([data.parse_clip(c) for c in inv.clip_id])
inv = pd.concat([inv.reset_index(drop=True), parsed], axis=1)
if not FROZEN:
    inv.to_csv(safe(OUT / 'interim' / 'inventory.csv'), index=False, lineterminator='\n')
print('dataset root:', ROOT, '| clips:', len(inv), '| errors:', int((inv.error != '').sum()))

In [ ]:
gates=[]
def gate(name,got,expected,ok=None):
    passed=(got==expected) if ok is None else bool(ok)
    gates.append({'gate':name,'expected':expected,'got':got,'ok':passed})
if FROZEN:
    print('Historical full-dataset gates are not rerun after freeze; training-only verification active.')
else:
    gates = []
    def gate(name, got, expected, ok=None):
        passed = got == expected if ok is None else bool(ok)
        gates.append({'gate':name, 'expected':expected, 'got':got, 'ok':passed})
    gate('G1 total/folder counts', [len(inv), int((inv.folder=='yes_drone').sum()), int((inv.folder=='unknown').sum())], [11704,1332,10372])
    gate('G2 format and decode', [inv.sr.nunique(), sorted(inv.channels.unique().tolist()), sorted(inv.subtype.unique().tolist()), int(inv.truncated.sum()), int((inv.error!='').sum())], [1,[1],['PCM_16'],0,0], inv.sr.eq(16000).all() and inv.channels.eq(1).all() and inv.subtype.eq('PCM_16').all() and not inv.truncated.any() and not (inv.error!='').any())
    unknown = inv[inv.folder=='unknown']; drone = inv[inv.folder=='yes_drone']
    gate('G3 unknown styles', unknown['style'].value_counts().to_dict(), {'esc':10000,'silence':301,'other':71})
    gate('G4 drone types', drone['drone_type'].value_counts().to_dict(), {'bebop':666,'membo':666})
    gate('G5 exact zeros', int(inv.all_zero.sum()), 1089)
    zero_sizes = sorted(inv[inv.all_zero].groupby('pcm_sha1').size().tolist(), reverse=True)
    gate('G6 zero-content multiplicities', zero_sizes, [330,308,191,150,57,35,15,1,1,1])
    gate('G7 zero-containing/all-zero ESC groups', [int(inv[(inv['style']=='esc') & inv.all_zero].group_raw.nunique()), int(inv[inv['style']=='esc'].groupby('group_raw').all_zero.all().sum())], [332,0])
    nonzero = inv[~inv.all_zero]
    gate('G8 duplicate nonzero PCM contents', int((nonzero.groupby('pcm_sha1').size()>1).sum()), 0)
    drone_rows=inv[inv['style']=='drone']; extra=drone_rows.family=='extra_membo_D2'
    raw_recordings=int(drone_rows.recording.nunique()+extra.sum()-drone_rows.loc[extra,'recording'].nunique())
    gate('G9 raw drone recordings / ESC groups', [raw_recordings,int(inv[inv['style']=='esc'].group_raw.nunique())], [312,1528])
    gate('G10 group keys shared across labels', int(inv.groupby('group_raw').folder.nunique().gt(1).sum()), 0)
    gate('G11 clipped clips', [int((unknown.clipped_frac>.01).sum()),int((drone.clipped_frac>.01).sum())], [206,0])
    gate('G12 DC clips', [int((unknown.dc.abs()>.01).sum()),int((drone.dc.abs()>.01).sum())], [116,0])
    near = (~inv.all_zero) & (inv.rms_dbfs < -80)
    gate('G13 near-silent clips', [int((near & (inv.folder=='unknown')).sum()),int((near & (inv.folder=='yes_drone')).sum())], [57,2])
    partial = (~inv.all_zero) & (inv.zero_frac>.5)
    gate('G14 partial-zero clips', [int((partial & (inv.folder=='unknown')).sum()),int((partial & (inv.folder=='yes_drone')).sum())], [244,2])
    med = inv[~inv.all_zero].groupby('folder').rms_dbfs.median()
    gate('G15 median nonzero RMS', [round(float(med['unknown']),1),round(float(med['yes_drone']),1)], [-24.8,-24.6], abs(float(med['unknown'])+24.8)<=.1 and abs(float(med['yes_drone'])+24.6)<=.1)
    L = data.l_rule_table(inv)
    L.to_csv(safe(PROJECT/'results'/'tables'/'audit_L_rule.csv'),index=False,lineterminator='\n')
    short = inv[inv.n_samples < 3968].clip_id.tolist()
    gate('G16 L=0.5 short clips', sorted(short), ['unknown/doing_the_dishes019.wav','unknown/silence300.wav'])
    gates_df = pd.DataFrame(gates); gates_df.to_csv(safe(OUT/'interim'/'audit_gates.csv'),index=False,lineterminator='\n')
    display(gates_df)


In [ ]:
if FROZEN:
    print('Multiclass audit skipped after freeze; no validation/test or multiclass audio opened.')
else:
    # One-time cross-folder PCM identity audit; do not read Multiclass_Drone_Audio after this cell.
    multi_base = ROOT / 'Multiclass_Drone_Audio'
    multi_hashes = set()
    for folder in ('unknown','bebop_1','membo_1'):
        for wav in sorted((multi_base/folder).glob('*.wav')): multi_hashes.add(data.clip_stats(wav)['pcm_sha1'])
    binary_match = int(inv.pcm_sha1.isin(multi_hashes).sum())
    gate('G17 binary PCM identity in multiclass', binary_match, 11704)
    # Keep only aggregate verdict. Multiclass content is not read again.
    pd.DataFrame(gates).to_csv(safe(OUT/'interim'/'audit_gates.csv'),index=False,lineterminator='\n')
    print('Multiclass identity:', binary_match, '/', len(inv))


In [ ]:
# G18: checked sample statistics, then measure the soxr HQ transition edge before building caches.
# Before freeze, the audit may use the full inventory. After freeze, only the train manifest is plotted.
from dronedet.plotting import audit_figures, train_only_audit_figures
_frozen_path=PROJECT/'data/processed/splits'/'FROZEN.txt'
if _frozen_path.exists():
 _audit_plots=[]  # Historical figures stay untouched during post-freeze verification.
 print('Train-only audit figures:',[p.name for p in _audit_plots])
else:
 _audit_plots=audit_figures(inv,safe(PROJECT/'results'/'figures'),window_lengths=(7936,15872,23808,32000))
 print('Pre-freeze audit figures:',[p.name for p in _audit_plots])
SPOT={'unknown/1-137-A-320.wav':(16347,-40.5,.06),'yes_drone/B_S2_D1_067-bebop_000_.wav':(16384,-17.5,.06),'unknown/1-1791-A-263.wav':(16347,-240.,0.),'yes_drone/B_S2_D1_069-bebop_002_.wav':(16384,-22.35,.06),'yes_drone/B_S2_D1_067-bebop_003_.wav':(16384,-21.85,.06),'yes_drone/extra_membo_D2_2011.wav':(16043,-41.25,.06)}
for cid,(n,db,tol) in SPOT.items():
 if FROZEN and cid not in set(inv.clip_id):
  continue  # Historical examples outside train are not inspected.
 row=inv.set_index('clip_id').loc[cid]; ok=int(row.n_samples)==n and abs(float(row.rms_dbfs)-db)<=tol
 gate('G18 spot '+cid, bool(ok), True)
from dronedet.data_preparation import record_resampler_measurement
edge=record_resampler_measurement(PROJECT/'config.yaml',safe(OUT/'interim'/'passband_verification.json'))
from dronedet.config import load_config
measured_fe=load_config(PROJECT/'config.yaml'); validation=measured_fe.validate_filterbank()
gate('G19 measured resampler F_MAX', [edge['f1db'],edge['fmax']], [measured_fe.measured_passband_hz,measured_fe.fmax], edge['status']=='PASS')
if edge['status']!='PASS':
 raise RuntimeError('Passband differs from frozen configuration; measurement recorded, config not changed')
# Record the local archive timestamp and infer its source commit only when the archive hash matches.
from dronedet.config import load_config, config_hash
zip_path=P['raw_root'].with_suffix('.zip')
from datetime import datetime
archive_created=datetime.fromtimestamp(zip_path.stat().st_ctime).astimezone().isoformat() if zip_path.is_file() else None
download_date=(archive_created+' (local archive creation timestamp; actual download time unverified)') if archive_created else None
source_commit='ec6139b3e17b9fe4b19eb54a7d6f4cf0c8e4b3c9'
source_zip_sha='90d09a04ce08bd08f0e76d92d0aebd394346317460dad4bdc8344c2ea9683f37'
source_basis='Inferred from the matching archive timestamps and matching README.md/LICENSE.md Git blob hashes; commit is retained only when the archive SHA-256 matches.'
log=data.environment_log(OUT/'interim'/('environment_verification.json' if FROZEN else 'environment_log.json'),dataset_sha256=None,download_date=download_date,repository_commit=source_commit,source_repository='https://github.com/saraalemadi/DroneAudioDataset',repository_commit_zip_sha256=source_zip_sha,repository_commit_basis=source_basis,frontend_params_hash=config_hash(load_config(PROJECT/'config.yaml')),zip_path=zip_path)
gate('G20 environment log written',(OUT/'interim'/('environment_verification.json' if FROZEN else 'environment_log.json')).is_file(),True)
gates_df=pd.DataFrame(gates); gates_df.to_csv(safe(OUT/'interim'/('audit_verification_gates.csv' if FROZEN else 'audit_gates.csv')),index=False,lineterminator='\n')
print('G19 F_MAX:',edge['fmax'],'Hz; -1 dB edge:',edge['f1db'],'Hz; filterbank:',validation)
print('Gates:',gates_df.ok.value_counts().to_dict())
print('G20 environment log:',log)
print('Decision log draft: D015 uses log-swept chirps through soxr HQ from both 44.1/48 kHz to 16 kHz and records the lower edge and chooses the largest 250 Hz multiple below the first -1 dB point; G9 counts 312 raw drone recordings and 1,528 ESC groups; D040 combines 56 extra clips into one pinned split group.')


In [ ]:
# Read the provenance record written by the audit cell; do not replace it with incomplete defaults.
log=json.loads((PROJECT/'data/interim'/('environment_verification.json' if FROZEN else 'environment_log.json')).read_text(encoding='utf-8'))
print(json.dumps(log,indent=2))
print('Decision log draft: Use seed 42 fixed before splitting; L=0.5 s gives a 7,936-sample, 28-frame window and reproduces the ≥95% full-window rule; inventory rows are sorted relative POSIX paths and PCM fingerprints hash channel-0 int16 samples.')
